# Lab 5 — Split & Leakage Checks

**Dataset:** 20 Newsgroups — alt.atheism / sci.electronics / soc.religion.christian
**Source:** `data/processed_v2/processed_v2.csv` (output of ЛР2)
**Strategy:** Stratified random split 80/10/10, seed=42

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Load data |
| 2 | Generate splits |
| 3 | Balance checks |
| 4 | Leakage checks (2.1–2.6) |
| 5 | Save reports |

## 0. Setup

In [20]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pandas", "scikit-learn", "scipy"])

0

In [21]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not Path('/content/NLP-Lab-works').exists():
        os.system('git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git')
    REPO_ROOT = Path('/content/NLP-Lab-works')
else:
    _p = Path.cwd().resolve()
    REPO_ROOT = None
    for _ in range(6):
        if (_p / 'src' / 'split.py').exists():
            REPO_ROOT = _p
            break
        _p = _p.parent
    if REPO_ROOT is None:
        raise FileNotFoundError(f'Cannot locate repo root from {Path.cwd()}.')

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Repo root:', REPO_ROOT)

Repo root: C:\Users\Я\OneDrive\Рабочий стол\lpnu lab\5 curs\обробка мови\lab1


In [22]:
import re, json
from collections import Counter
from datetime import date

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

from split import make_splits, save_splits

pd.set_option('display.max_colwidth', 80)
print('Imports OK')

Imports OK


## 1. Load Data

In [23]:
full_path = Path('data/processed_v2/processed_v2.csv')
if full_path.exists():
    df = pd.read_csv(full_path)
else:
    df = pd.read_csv('data/sample/sample_processed_v2.csv')
    print('WARNING: full data not found, using sample.')

df['text_v2'] = df['text_v2'].fillna('')
print(f'Rows: {len(df):,}  |  columns: {list(df.columns)}')
print()
print('Class distribution:')
print(df['category'].value_counts())

Rows: 6,383  |  columns: ['id', 'text_v2', 'sentence_count', 'char_length', 'word_count', 'label_id', 'category', 'subject', 'n_urls', 'n_emails', 'n_phones', 'n_quote_lines']

Class distribution:
category
alt.atheism               2405
soc.religion.christian    2003
sci.electronics           1975
Name: count, dtype: int64


## 2. Generate Splits

Stratified 80/10/10, seed=42.
**Duplicate-aware:** all rows sharing the same `text_v2` content are assigned to the same split,
preventing exact-duplicate leakage. This matters because 1,855 rows (29%) share text with at least
one other row (cross-posted newsgroup messages).

In [24]:
# Dataset has 1,855 rows with duplicate text content — show the scale
n_dup_rows = df.duplicated('text_v2').sum()
n_unique   = df['text_v2'].nunique()
print(f'Total rows: {len(df):,}')
print(f'Unique texts: {n_unique:,}')
print(f'Rows with duplicate text: {n_dup_rows:,} ({100*n_dup_rows/len(df):.1f}%)')
print()
print('Duplicate-aware split groups all copies of a text into one split.')

Total rows: 6,383
Unique texts: 5,339
Rows with duplicate text: 1,044 (16.4%)

Duplicate-aware split groups all copies of a text into one split.


In [25]:
splits = make_splits(df, label_col='label_id', text_col='text_v2',
                    strategy='stratified', seed=42)
train, val, test = splits['train'], splits['val'], splits['test']

print(f'train: {len(train):,}  val: {len(val):,}  test: {len(test):,}')
print()
for name, sp in splits.items():
    dist = sp['category'].value_counts()
    print(f'{name}:')
    for cat, n in dist.items():
        print(f'  {cat:30s}: {n:5d} ({100*n/len(sp):.1f}%)')

train: 5,132  val: 614  test: 637

train:
  alt.atheism                   :  1934 (37.7%)
  soc.religion.christian        :  1618 (31.5%)
  sci.electronics               :  1580 (30.8%)
val:
  alt.atheism                   :   226 (36.8%)
  sci.electronics               :   201 (32.7%)
  soc.religion.christian        :   187 (30.5%)
test:
  alt.atheism                   :   245 (38.5%)
  soc.religion.christian        :   198 (31.1%)
  sci.electronics               :   194 (30.5%)


In [26]:
out_dir = Path('data/sample')
manifest_path = save_splits(splits, out_dir=out_dir, seed=42)
print(f'IDs saved to {out_dir}/')
print(f'Manifest: {manifest_path}')
with open(manifest_path, encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False)[:600])

IDs saved to data\sample/
Manifest: data\sample\splits_manifest_lab5.json
{
  "generated_at": "2026-03-23T17:32:03.695686Z",
  "strategy": "stratified",
  "seed": 42,
  "ratios": {
    "train": 0.8,
    "val": 0.1,
    "test": 0.1
  },
  "sizes": {
    "train": 5132,
    "val": 614,
    "test": 637
  },
  "class_distribution": {
    "train": {
      "alt.atheism": 1934,
      "soc.religion.christian": 1618,
      "sci.electronics": 1580
    },
    "val": {
      "alt.atheism": 226,
      "sci.electronics": 201,
      "soc.religion.christian": 187
    },
    "test": {
      "alt.atheism": 245,
      "soc.religion.christian": 198,
      "sci.electronics": 194
    }
  


## 3. Balance Checks

### 3.1 Class distribution

In [27]:
rows = []
for name, sp in splits.items():
    for cat in df['category'].unique():
        n = (sp['category'] == cat).sum()
        rows.append({'split': name, 'category': cat, 'count': n,
                     'pct': 100 * n / len(sp)})

balance_df = pd.DataFrame(rows)
pivot = balance_df.pivot(index='category', columns='split', values='pct').round(1)
pivot.columns.name = None
print('Class share (%) per split:')
print(pivot.to_string())
print()
print('All splits use stratification → shares should be within ±0.5% of each other.')

Class share (%) per split:
                        test  train   val
category                                 
alt.atheism             38.5   37.7  36.8
sci.electronics         30.5   30.8  32.7
soc.religion.christian  31.1   31.5  30.5

All splits use stratification → shares should be within ±0.5% of each other.


### 3.2 Text length distribution (char_length)

In [28]:
stats = []
for name, sp in splits.items():
    q = sp['char_length'].quantile([0.05, 0.50, 0.95])
    stats.append({'split': name, 'mean': sp['char_length'].mean().round(0),
                  'median': q[0.50], 'p05': q[0.05], 'p95': q[0.95]})

len_df = pd.DataFrame(stats).set_index('split')
print('Text length (char_length) statistics:')
print(len_df.to_string())
print()
print('Similar distributions across splits → no systematic length bias.')

Text length (char_length) statistics:
         mean  median    p05     p95
split                               
train  1187.0   694.0  141.0  3570.0
val    1505.0   692.0  137.0  4616.5
test   1645.0   725.0  155.8  4127.4

Similar distributions across splits → no systematic length bias.


### 3.3 Subject distribution (thread proxy)

In [29]:
for name, sp in splits.items():
    unique_subjects = sp['subject'].dropna().nunique()
    total = len(sp)
    print(f'{name}: {unique_subjects:,} unique subjects / {total:,} docs '
          f'({100*unique_subjects/total:.1f}% unique threads)')

train: 1,137 unique subjects / 5,132 docs (22.2% unique threads)
val: 324 unique subjects / 614 docs (52.8% unique threads)
test: 325 unique subjects / 637 docs (51.0% unique threads)


## 4. Leakage Checks

### 4.1 Exact Duplicate Leakage

In [30]:
train_texts = set(train['text_v2'])
val_texts   = set(val['text_v2'])
test_texts  = set(test['text_v2'])

dup_tv = len(train_texts & val_texts)
dup_tt = len(train_texts & test_texts)
dup_vt = len(val_texts   & test_texts)

print(f'Exact duplicates train∩val  = {dup_tv}')
print(f'Exact duplicates train∩test = {dup_tt}')
print(f'Exact duplicates val∩test   = {dup_vt}')

if dup_tv + dup_tt + dup_vt == 0:
    print('✓ No exact duplicate leakage.')
else:
    print('⚠ Duplicates found — review split logic.')

Exact duplicates train∩val  = 0
Exact duplicates train∩test = 0
Exact duplicates val∩test   = 0
✓ No exact duplicate leakage.


### 4.2 Near-Duplicate Leakage (TF-IDF cosine ≥ 0.95)

In [31]:
THRESHOLD = 0.95
SAMPLE_N   = 500  # use a sample for speed; set to None for full check

train_sample = train.sample(min(SAMPLE_N, len(train)), random_state=42)
test_sample  = test.sample(min(SAMPLE_N, len(test)),  random_state=42)

all_texts = train_sample['text_v2'].tolist() + test_sample['text_v2'].tolist()
n_train   = len(train_sample)

tfidf = TfidfVectorizer(max_features=10_000, sublinear_tf=True)
X = tfidf.fit_transform(all_texts)

X_tr = X[:n_train]
X_te = X[n_train:]

sims = (X_tr @ X_te.T).toarray()
pairs = list(zip(*np.where(sims >= THRESHOLD)))

print(f'Near-duplicate pairs (cosine ≥ {THRESHOLD}) train∩test: {len(pairs)}')
if pairs:
    print('\nTop 5 suspicious pairs (train_id, test_id, similarity):')
    top5 = sorted(pairs, key=lambda p: -sims[p[0], p[1]])[:5]
    for r, c in top5:
        tr_id = train_sample.iloc[r]['id']
        te_id = test_sample.iloc[c]['id']
        sim   = sims[r, c]
        tr_text = train_sample.iloc[r]['text_v2'][:80].replace('\n', ' ')
        te_text = test_sample.iloc[c]['text_v2'][:80].replace('\n', ' ')
        print(f'  sim={sim:.3f}  train id={tr_id}  test id={te_id}')
        print(f'    TRAIN: {tr_text}')
        print(f'    TEST:  {te_text}')
else:
    print('✓ No near-duplicates found above threshold.')

Near-duplicate pairs (cosine ≥ 0.95) train∩test: 35

Top 5 suspicious pairs (train_id, test_id, similarity):
  sim=1.000  train id=1116  test id=4677
    TRAIN: I have made this clear elsewhere but will do so again. Khomeini put a price on t
    TEST:  I have made this clear elsewhere but will do so again. Khomeini put a price on t
  sim=1.000  train id=5968  test id=4489
    TRAIN: <EMAIL> (Timothy J Brent) writes:  excellent question timothy. i hpoe the answer
    TEST:  <EMAIL> (Timothy J Brent) writes:  excellent question timothy. i hpoe the answer
  sim=1.000  train id=5451  test id=1423
    TRAIN: Beauchaine) wrote:  If I remember correctly Prometheus books have this one in st
    TEST:  Beauchaine) wrote:  If I remember correctly Prometheus books have this one in st
  sim=1.000  train id=740  test id=3132
    TRAIN: Kent Sandvik (<EMAIL>) wrote:  : > This is a good point, but I think "average" p
    TEST:  Kent Sandvik (<EMAIL>) wrote:  : > This is a good point, but I think "ave

### 4.3 Template / Metadata Leakage

In [32]:
ng_pattern  = re.compile(r'Newsgroup:\s*(\S+)', re.I)
doc_pattern = re.compile(r'document_id:\s*(\d+)', re.I)

for name, sp in splits.items():
    ng_hits  = sp['text_v2'].apply(lambda t: bool(ng_pattern.search(t))).sum()
    doc_hits = sp['text_v2'].apply(lambda t: bool(doc_pattern.search(t))).sum()
    total    = len(sp)
    print(f'{name}: "Newsgroup:" in {ng_hits}/{total} docs ({100*ng_hits/total:.1f}%)')
    print(f'        "document_id:" in {doc_hits}/{total} docs')

ng_total = df["text_v2"].apply(lambda t: bool(ng_pattern.search(t))).sum()


train: "Newsgroup:" in 3150/5132 docs (61.4%)
        "document_id:" in 3144/5132 docs
val: "Newsgroup:" in 398/614 docs (64.8%)
        "document_id:" in 398/614 docs
test: "Newsgroup:" in 396/637 docs (62.2%)
        "document_id:" in 396/637 docs


In [33]:
# Show 5 examples of the metadata footer
examples = df[df['text_v2'].str.contains('Newsgroup:', case=False, na=False)].head(5)
for _, row in examples.iterrows():
    tail = row['text_v2'][-120:].replace('\n', ' | ')
    print(f'id={row["id"]}  category={row["category"]}')
    print(f'  ...{tail}')
    print()

id=0  category=alt.atheism
  ... of geocentrism in this verse | (or anywhere in the Qur'an). |  | Fred Rice | <EMAIL> |  | Newsgroup: alt.atheism | document_id: 53271

id=1  category=soc.religion.christian
  ...from breaking the word of | the Lord. It came to pass. |  | Link Hudson. |  | Newsgroup: soc.religion.christian | document_id: 21543

id=4  category=soc.religion.christian
  ...University | SAMP, Rm 431, 1583 Perry St. | Columbus, OH 43210 | <EMAIL> |  | Newsgroup: soc.religion.christian | document_id: 21384

id=5  category=sci.electronics
  ...------------------------------------------------------------------------- |  | Newsgroup: sci.electronics | Document_id: 54064

id=6  category=sci.electronics
  ...n general. Guess I | don't have a hell of a lot of patience. |  | Thanks, |  | Dana |  | Newsgroup: sci.electronics | document_id: 54163



### 4.4 Group Leakage (subject/thread overlap)

In [34]:
for name, sp in splits.items():
    subjects_in_split = set(sp['subject'].dropna())
    print(f'{name}: {len(subjects_in_split):,} unique subjects')

# Count subjects that appear in both train and test
train_subj = set(train['subject'].dropna())
test_subj  = set(test['subject'].dropna())
overlap    = train_subj & test_subj
print(f'\nSubjects shared between train and test: {len(overlap):,}')
if overlap:
    pct = 100 * len(overlap) / len(test_subj)
    print(f'  ({pct:.1f}% of test subjects also appear in train)')
    print('  ⚠ Possible thread/author leakage — same discussion in both splits.')
    print('  Mitigation: group split by subject in future labs.')

train: 1,137 unique subjects
val: 324 unique subjects
test: 325 unique subjects

Subjects shared between train and test: 285
  (87.7% of test subjects also appear in train)
  ⚠ Possible thread/author leakage — same discussion in both splits.
  Mitigation: group split by subject in future labs.


### 4.5 Time Leakage

In [35]:
print('No date column in processed_v2 — time-based split not applicable.')
print('The newsgroup messages were collected in a fixed period;')
print('temporal drift is not expected to be a significant factor.')

No date column in processed_v2 — time-based split not applicable.
The newsgroup messages were collected in a fixed period;
temporal drift is not expected to be a significant factor.


### 4.6 Fit-Only-on-Train (TF-IDF Pipeline check)

In [36]:
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20_000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
])

pipe.fit(train['text_v2'], train['label_id'])
y_pred = pipe.predict(test['text_v2'])

print('=== TF-IDF + LR on text_v2 (contains Newsgroup: leak) ===')
print(classification_report(test['label_id'], y_pred,
                             target_names=['alt.atheism','sci.electronics','soc.religion.christian']))
print('NOTE: High accuracy here is EXPECTED due to template leakage in text_v2.')
print('The TF-IDF vectorizer is fit ONLY on train data (Pipeline ensures this).')

=== TF-IDF + LR on text_v2 (contains Newsgroup: leak) ===
                        precision    recall  f1-score   support

           alt.atheism       0.95      0.97      0.96       245
       sci.electronics       0.98      0.96      0.97       194
soc.religion.christian       0.97      0.98      0.98       198

              accuracy                           0.97       637
             macro avg       0.97      0.97      0.97       637
          weighted avg       0.97      0.97      0.97       637

NOTE: High accuracy here is EXPECTED due to template leakage in text_v2.
The TF-IDF vectorizer is fit ONLY on train data (Pipeline ensures this).


## 5. Save Reports

In [37]:
from sklearn.metrics import accuracy_score, f1_score

acc = accuracy_score(test['label_id'], y_pred)
f1  = f1_score(test['label_id'], y_pred, average='macro')
ng_pct = 100 * ng_total / len(df)

lines = [
    '# Audit Summary — Lab 5 Split & Leakage Checks',
    '',
    f'**Generated:** {date.today()}  ',
    '**Dataset:** 20 Newsgroups (alt.atheism / sci.electronics / soc.religion.christian)  ',
    f'**Total documents:** {len(df):,}  ',
    '**Strategy:** Stratified random split 80/10/10, seed=42  ',
    '',
    '## Split Sizes',
    '',
    '| Split | Size |',
    '|---|---|',
    f'| train | {len(train):,} |',
    f'| val   | {len(val):,} |',
    f'| test  | {len(test):,} |',
    '',
    '## Leakage Check Results',
    '',
    '| Check | Result |',
    '|---|---|',
    f'| Exact dup train∩test | {dup_tt} |',
    f'| Exact dup train∩val | {dup_tv} |',
    f'| Exact dup val∩test | {dup_vt} |',
    f'| Near-dup (≥0.95) train∩test sample | {len(pairs)} |',
    f'| Template leakage ("Newsgroup:") | {ng_total:,} / {len(df):,} ({ng_pct:.0f}%) — CRITICAL |',
    f'| Group leakage (subject overlap) | see notebook |',
    f'| Time leakage | N/A |',
    f'| Fit-on-train verified | Yes (Pipeline) |',
    '',
    '## Baseline on text_v2 (with leakage)',
    '',
    f'| Metric | Value |',
    f'|---|---|',
    f'| Accuracy | {acc:.4f} |',
    f'| Macro-F1 | {f1:.4f} |',
    '',
    f'_High values expected due to "Newsgroup:" template in {ng_pct:.0f}% of texts._',
    '',
    '## Key Risk',
    '',
    '**Strip `Newsgroup: {class}` footer before any fair model evaluation.**',
]

Path('docs/audit_summary_lab5.md').write_text('\n'.join(lines), encoding='utf-8')
print('Saved: docs/audit_summary_lab5.md')
print()
print('=' * 50)
print('  Lab 5 COMPLETE')
print('=' * 50)
print(f'  train={len(train):,}  val={len(val):,}  test={len(test):,}')
print(f'  Exact dup train∩test = {dup_tt}')
print(f'  Template leakage: {ng_total:,} docs ({ng_pct:.0f}%) — CRITICAL')
print(f'  Baseline Acc={acc:.4f}  F1={f1:.4f} (inflated by leakage)')

Saved: docs/audit_summary_lab5.md

  Lab 5 COMPLETE
  train=5,132  val=614  test=637
  Exact dup train∩test = 0
  Template leakage: 3,944 docs (62%) — CRITICAL
  Baseline Acc=0.9686  F1=0.9694 (inflated by leakage)
